# Stacking: CatBoost + DeBERTa-v3

Two approaches for combining handcrafted features with a fine-tuned DeBERTa-v3 cross-encoder.

**Approach 1 – True Stacking**

1. CatBoost on 20 handcrafted features (HP-tuned via Optuna)
2. Fine-tuned DeBERTa-v3 cross-encoder OOF predictions
3. Meta-model (LR / CatBoost) on stacked OOF predictions

**Approach 2 – CLS + Features MLP**

1. Extract `[CLS]` token embeddings from DeBERTa
2. Concatenate with 20 handcrafted features
3. Train a small MLP on the combined representation

All experiments use **5-fold stratified cross-validation** (random_state=42).


In [1]:
import gc
import json
import re
import string
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    AutoConfig,
    AutoTokenizer,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
N_OPTUNA_TRIALS = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
DATA_DIR = Path("../data/raw")
ARTIFACTS_DIR = Path("../artifacts/stacking")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DEBERTA_MODEL_DIR = (
    Path("../models")
    / "deberta-v3-base-cross-encoder-1024-20260909T175551Z-1-001"
    / "deberta-v3-base-cross-encoder-1024"
)

WORD_RE = re.compile(r"\b\w+\b", re.UNICODE)
NUMBER_RE = re.compile(r"(?<!\w)[+-]?\d+(?:[.,]\d+)?%?(?!\w)")
REFUSAL = re.compile(
    r"I cannot|I can't|I'm sorry|As an AI|I am unable|I apologize",
    re.I,
)

tr = pd.read_csv(DATA_DIR / "train.csv")
te = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {tr.shape}, Test: {te.shape}")

Train: (57477, 9), Test: (3, 4)


In [3]:
def parse_text(s):
    try:
        return " ".join(x or "" for x in json.loads(s))
    except Exception:
        return str(s)


for col in ["prompt", "response_a", "response_b"]:
    tr[col + "_txt"] = tr[col].apply(parse_text)
    te[col + "_txt"] = te[col].apply(parse_text)


y = np.argmax(
    tr[["winner_model_a", "winner_model_b", "winner_tie"]].values,
    axis=1,
)

print(f"Labels distribution: {np.bincount(y)}")

Labels distribution: [20064 19652 17761]


In [4]:
def _get_words(text):
    if not isinstance(text, str):
        return []
    return WORD_RE.findall(text.lower())


def _get_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


def _get_paragraphs(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return [p.strip() for p in re.split(r"\n+", text.strip()) if p.strip()]


def _paragraph_features(text):
    paragraphs = _get_paragraphs(text)
    words_per_para = [len(_get_words(p)) for p in paragraphs]
    sents_per_para = [len(_get_sentences(p)) for p in paragraphs]
    n = len(paragraphs)
    mean_pl = float(np.mean(words_per_para)) if words_per_para else 0.0
    sd_pl = float(np.std(words_per_para)) if words_per_para else 0.0
    avg_spp = float(np.mean(sents_per_para)) if sents_per_para else 0.0
    return n, mean_pl, sd_pl, avg_spp


def _repetition_density(text):
    words = _get_words(text)
    if not words:
        return 0.0
    return 1.0 - len(set(words)) / len(words)


def _punctuation_density(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(c in string.punctuation for c in text) / len(text)


def _has_numbers(text):
    return 1.0 if NUMBER_RE.search(str(text)) else 0.0


def get_handcrafted_features(df: pd.DataFrame) -> np.ndarray:
    refusals_a = df.response_a_txt.str.count(REFUSAL).to_numpy(float)
    refusals_b = df.response_b_txt.str.count(REFUSAL).to_numpy(float)
    prompt_length = df.prompt_txt.str.len().to_numpy(float) / 1000.0

    para_a = np.array([_paragraph_features(t) for t in df.response_a_txt])
    para_b = np.array([_paragraph_features(t) for t in df.response_b_txt])

    rep_a = np.array([_repetition_density(t) for t in df.response_a_txt])
    rep_b = np.array([_repetition_density(t) for t in df.response_b_txt])

    punct_a = np.array([_punctuation_density(t) for t in df.response_a_txt])
    punct_b = np.array([_punctuation_density(t) for t in df.response_b_txt])

    length_a = df.response_a_txt.str.len().to_numpy(float)
    length_b = df.response_b_txt.str.len().to_numpy(float)
    len_ratio = np.log1p(length_a) - np.log1p(length_b)

    has_num_a = df.response_a_txt.apply(lambda x: _has_numbers(x)).to_numpy()
    has_num_b = df.response_b_txt.apply(lambda x: _has_numbers(x)).to_numpy()

    return np.column_stack(
        [
            refusals_a,
            refusals_b,
            prompt_length,
            para_a[:, 0],
            para_a[:, 1],
            para_a[:, 2],
            para_a[:, 3],
            para_b[:, 0],
            para_b[:, 1],
            para_b[:, 2],
            para_b[:, 3],
            rep_a,
            rep_b,
            punct_a,
            punct_b,
            len_ratio,
            length_a,
            length_b,
            has_num_a,
            has_num_b,
        ]
    )


FEATURE_NAMES_20 = [
    "refusals_a",
    "refusals_b",
    "prompt_length",
    "num_paragraphs_a",
    "mean_paragraph_length_a",
    "sd_paragraph_length_a",
    "avg_sent_per_paragraph_a",
    "num_paragraphs_b",
    "mean_paragraph_length_b",
    "sd_paragraph_length_b",
    "avg_sent_per_paragraph_b",
    "repetition_density_a",
    "repetition_density_b",
    "punctuation_density_a",
    "punctuation_density_b",
    "len_ratio_a_b",
    "length_a",
    "length_b",
    "has_numbers_a",
    "has_numbers_b",
]

X_20 = get_handcrafted_features(tr)
print(f"Handcrafted features: {X_20.shape}")

Handcrafted features: (57477, 20)


In [5]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
CV_SPLITS = list(skf.split(X_20, y))

print(f"CV folds: {len(CV_SPLITS)}")
for i, (tr_idx, va_idx) in enumerate(CV_SPLITS):
    print(f"  Fold {i}: train={len(tr_idx)}, val={len(va_idx)}")

CV folds: 5
  Fold 0: train=45981, val=11496
  Fold 1: train=45981, val=11496
  Fold 2: train=45982, val=11495
  Fold 3: train=45982, val=11495
  Fold 4: train=45982, val=11495


---

## 1. CatBoost HP Tuning (Optuna)

Tune CatBoost hyperparameters on the 20 handcrafted features using 5-fold
stratified CV. Objective: minimize mean OOF log loss.


In [ ]:
def catboost_objective(trial):
    params = {
        "loss_function": "MultiClass",
        "classes_count": 3,
        "iterations": trial.suggest_int("iterations", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_seed": SEED,
        "verbose": False,
        "allow_writing_files": False,
        "task_type": "GPU",
    }

    oof = np.zeros((len(y), 3))

    for train_idx, val_idx in CV_SPLITS:
        model = CatBoostClassifier(**params)
        model.fit(X_20[train_idx], y[train_idx])
        oof[val_idx] = model.predict_proba(X_20[val_idx])

    return log_loss(y, oof)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(catboost_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nBest trial:")
print(f"  Log loss: {study.best_trial.value:.6f}")
print(f"  Params:   {study.best_trial.params}")

  0%|          | 0/20 [00:00<?, ?it/s]


Best trial:
  Log loss: 1.027952
  Params:   {'iterations': 896, 'learning_rate': 0.028869220380495747, 'depth': 7, 'l2_leaf_reg': 5.920392514089517, 'border_count': 73}


### 1b. CatBoost OOF with best hyperparameters


In [8]:
best_cb_params = study.best_trial.params
best_cb_params["loss_function"] = "MultiClass"
best_cb_params["classes_count"] = 3
best_cb_params["random_seed"] = SEED
best_cb_params["verbose"] = False
best_cb_params["allow_writing_files"] = False

catboost_oof = np.zeros((len(y), 3))
catboost_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(CV_SPLITS):
    model = CatBoostClassifier(**best_cb_params)
    model.fit(X_20[train_idx], y[train_idx])
    pred = model.predict_proba(X_20[val_idx])
    catboost_oof[val_idx] = pred
    score = log_loss(y[val_idx], pred)
    catboost_fold_scores.append(score)
    print(f"  Fold {fold}: {score:.4f}")

catboost_oof_loss = log_loss(y, catboost_oof)
print(
    f"\nCatBoost OOF log loss: {catboost_oof_loss:.4f} (+/- {np.std(catboost_fold_scores):.4f})"
)

np.save(ARTIFACTS_DIR / "catboost_oof_probs.npy", catboost_oof)
print(f"Saved: {ARTIFACTS_DIR / 'catboost_oof_probs.npy'}")

  Fold 0: 1.0308
  Fold 1: 1.0301
  Fold 2: 1.0253
  Fold 3: 1.0243
  Fold 4: 1.0288

CatBoost OOF log loss: 1.0279 (+/- 0.0026)
Saved: ../artifacts/stacking/catboost_oof_probs.npy


---

## 2. DeBERTa-v3 Cross-Encoder OOF Inference

Run the fine-tuned DeBERTa-v3 on the training data using the same CV splits.
Save OOF probability predictions and `[CLS]` token embeddings for each sample.


In [9]:
DEBERTA_MAX_LENGTH = 1024
DEBERTA_BATCH_SIZE = 16

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_DIR)
deberta_config = AutoConfig.from_pretrained(DEBERTA_MODEL_DIR)

PROMPT_PREFIX_IDS = deberta_tokenizer("PROMPT:\n", add_special_tokens=False)[
    "input_ids"
]
RESP_A_PREFIX_IDS = deberta_tokenizer("\n\nRESPONSE A:\n", add_special_tokens=False)[
    "input_ids"
]
RESP_B_PREFIX_IDS = deberta_tokenizer("\n\nRESPONSE B:\n", add_special_tokens=False)[
    "input_ids"
]

TAIL_FRACTION = 0.25
MAX_PROMPT_TOKENS = 256
SPECIAL_OVERHEAD = 1 + 3
PREFIX_OVERHEAD = (
    len(PROMPT_PREFIX_IDS) + len(RESP_A_PREFIX_IDS) + len(RESP_B_PREFIX_IDS)
)
CONTENT_BUDGET = DEBERTA_MAX_LENGTH - SPECIAL_OVERHEAD - PREFIX_OVERHEAD

print(f"Content budget: {CONTENT_BUDGET} tokens")

Content budget: 1011 tokens


In [10]:
def _head_tail_truncate(token_ids, budget, tail_fraction=TAIL_FRACTION):
    if budget <= 0:
        return []
    if len(token_ids) <= budget:
        return token_ids
    if budget == 1:
        return token_ids[:1]
    tail_size = max(1, int(round(budget * tail_fraction)))
    tail_size = min(tail_size, budget - 1)
    head_size = budget - tail_size
    return token_ids[:head_size] + token_ids[-tail_size:]


def _balanced_budgets(len_a, len_b, total):
    if len_a + len_b <= total:
        return len_a, len_b
    half = total // 2
    if len_a <= half:
        return len_a, min(len_b, total - len_a)
    if len_b <= half:
        return min(len_a, total - len_b), len_b
    if len_a >= len_b:
        return total - half, half
    return half, total - half


def _build_input(prompt_ids, resp_a_ids, resp_b_ids):
    prompt_budget = min(len(prompt_ids), MAX_PROMPT_TOKENS, CONTENT_BUDGET)
    resp_budget = CONTENT_BUDGET - prompt_budget
    a_budget, b_budget = _balanced_budgets(
        len(resp_a_ids), len(resp_b_ids), resp_budget
    )

    prompt_ids = _head_tail_truncate(prompt_ids, prompt_budget)
    resp_a_ids = _head_tail_truncate(resp_a_ids, a_budget)
    resp_b_ids = _head_tail_truncate(resp_b_ids, b_budget)

    cls_id = deberta_tokenizer.cls_token_id
    sep_id = deberta_tokenizer.sep_token_id

    input_ids = (
        [cls_id]
        + PROMPT_PREFIX_IDS
        + prompt_ids
        + [sep_id]
        + RESP_A_PREFIX_IDS
        + resp_a_ids
        + [sep_id]
        + RESP_B_PREFIX_IDS
        + resp_b_ids
        + [sep_id]
    )
    attention_mask = [1] * len(input_ids)

    assert len(input_ids) <= DEBERTA_MAX_LENGTH
    return input_ids, attention_mask


def _tokenize_raw(text):
    return deberta_tokenizer(
        text, add_special_tokens=False, truncation=False, padding=False
    )["input_ids"]


def prepare_cross_encoder_inputs(df):
    """Tokenize prompt/response texts and build smart-truncated inputs."""
    prompt_ids_list = [_tokenize_raw(t) for t in df["prompt_txt"]]
    resp_a_ids_list = [_tokenize_raw(t) for t in df["response_a_txt"]]
    resp_b_ids_list = [_tokenize_raw(t) for t in df["response_b_txt"]]

    all_input_ids = []
    all_attention_mask = []
    for pids, aids, bids in zip(prompt_ids_list, resp_a_ids_list, resp_b_ids_list):
        ids, mask = _build_input(pids, aids, bids)
        all_input_ids.append(ids)
        all_attention_mask.append(mask)

    return all_input_ids, all_attention_mask


print("Cross-encoder input preparation functions defined.")

Cross-encoder input preparation functions defined.


In [11]:
def run_deberta_full_oof(
    df,
    labels,
    cv_splits,
    model_dir,
    device,
    max_length=DEBERTA_MAX_LENGTH,
    batch_size=DEBERTA_BATCH_SIZE,
):
    """
    Run full DeBERTa OOF inference using
    AutoModelForSequenceClassification for probabilities
    and hidden states for [CLS] embeddings.
    """
    all_input_ids, all_attention_mask = prepare_cross_encoder_inputs(df)
    n = len(df)
    hidden_dim = 768

    oof_probs = np.zeros((n, 3), dtype=np.float32)
    oof_cls = np.zeros((n, hidden_dim), dtype=np.float32)

    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        print(f"\n=== DeBERTa OOF Fold {fold} ===")

        from transformers import AutoModelForSequenceClassification

        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        model.config.output_hidden_states = True
        model = model.to(device)
        model.eval()

        val_input_ids = [all_input_ids[i] for i in val_idx]
        val_attention_mask = [all_attention_mask[i] for i in val_idx]

        fold_probs = []
        fold_cls = []

        n_batches = (len(val_idx) + batch_size - 1) // batch_size

        for start in range(0, len(val_idx), batch_size):
            end = min(start + batch_size, len(val_idx))
            batch_ids = val_input_ids[start:end]
            batch_mask = val_attention_mask[start:end]

            max_len = max(len(ids) for ids in batch_ids)
            padded_ids = torch.zeros(
                len(batch_ids), max_len, dtype=torch.long, device=device
            )
            padded_mask = torch.zeros(
                len(batch_ids), max_len, dtype=torch.long, device=device
            )

            for i, (ids, mask) in enumerate(zip(batch_ids, batch_mask)):
                padded_ids[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)
                padded_mask[i, : len(mask)] = torch.tensor(mask, dtype=torch.long)

            with torch.no_grad():
                outputs = model(
                    input_ids=padded_ids,
                    attention_mask=padded_mask,
                )

            # Softmax probabilities
            logits = outputs.logits.cpu().numpy()
            probs = np.exp(logits - logits.max(axis=1, keepdims=True))
            probs = probs / probs.sum(axis=1, keepdims=True)
            fold_probs.append(probs)

            # [CLS] from last hidden state
            cls_emb = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
            fold_cls.append(cls_emb)

            if (start // batch_size) % 10 == 0:
                print(f"    batch {(start // batch_size) + 1}/{n_batches}")

        oof_probs[val_idx] = np.concatenate(fold_probs, axis=0)
        oof_cls[val_idx] = np.concatenate(fold_cls, axis=0)

        fold_loss = log_loss(labels[val_idx], oof_probs[val_idx])
        print(f"  Fold {fold} log loss: {fold_loss:.4f}")

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return oof_probs, oof_cls


print("Full DeBERTa OOF function defined.")

Full DeBERTa OOF function defined.


In [12]:
# Run DeBERTa OOF inference
# This will take a while (~30-60 min on GPU depending on hardware)
print("Starting DeBERTa OOF inference...")
print(f"Model: {DEBERTA_MODEL_DIR}")
print(f"Device: {DEVICE}")

deberta_oof_probs, deberta_oof_cls = run_deberta_full_oof(
    df=tr,
    labels=y,
    cv_splits=CV_SPLITS,
    model_dir=DEBERTA_MODEL_DIR,
    device=DEVICE,
)

deberta_oof_loss = log_loss(y, deberta_oof_probs)
print(f"\nDeBERTa OOF log loss: {deberta_oof_loss:.4f}")
print(f"CLS embeddings shape: {deberta_oof_cls.shape}")

# Save artifacts
np.save(ARTIFACTS_DIR / "deberta_oof_probs.npy", deberta_oof_probs)
np.save(ARTIFACTS_DIR / "deberta_oof_cls.npy", deberta_oof_cls)
print(f"Saved OOF probs and CLS embeddings to {ARTIFACTS_DIR}")

Starting DeBERTa OOF inference...
Model: ../models/deberta-v3-base-cross-encoder-1024-20260909T175551Z-1-001/deberta-v3-base-cross-encoder-1024
Device: cuda

=== DeBERTa OOF Fold 0 ===


W0909 22:23:33.320000 23036 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0909 22:23:33.356000 23036 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

    batch 1/719
    batch 11/719
    batch 21/719
    batch 31/719
    batch 41/719
    batch 51/719
    batch 61/719
    batch 71/719
    batch 81/719
    batch 91/719
    batch 101/719
    batch 111/719
    batch 121/719
    batch 131/719
    batch 141/719
    batch 151/719
    batch 161/719
    batch 171/719
    batch 181/719
    batch 191/719
    batch 201/719
    batch 211/719
    batch 221/719
    batch 231/719
    batch 241/719
    batch 251/719
    batch 261/719
    batch 271/719
    batch 281/719
    batch 291/719
    batch 301/719
    batch 311/719
    batch 321/719
    batch 331/719
    batch 341/719
    batch 351/719
    batch 361/719
    batch 371/719
    batch 381/719
    batch 391/719
    batch 401/719
    batch 411/719
    batch 421/719
    batch 431/719
    batch 441/719
    batch 451/719
    batch 461/719
    batch 471/719
    batch 481/719
    batch 491/719
    batch 501/719
    batch 511/719
    batch 521/719
    batch 531/719
    batch 541/719
    batch 551/719
   

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

    batch 1/719
    batch 11/719
    batch 21/719
    batch 31/719
    batch 41/719
    batch 51/719
    batch 61/719
    batch 71/719
    batch 81/719
    batch 91/719
    batch 101/719
    batch 111/719
    batch 121/719
    batch 131/719
    batch 141/719
    batch 151/719
    batch 161/719
    batch 171/719
    batch 181/719
    batch 191/719
    batch 201/719
    batch 211/719
    batch 221/719
    batch 231/719
    batch 241/719
    batch 251/719
    batch 261/719
    batch 271/719
    batch 281/719
    batch 291/719
    batch 301/719
    batch 311/719
    batch 321/719
    batch 331/719
    batch 341/719
    batch 351/719
    batch 361/719
    batch 371/719
    batch 381/719
    batch 391/719
    batch 401/719
    batch 411/719
    batch 421/719
    batch 431/719
    batch 441/719
    batch 451/719
    batch 461/719
    batch 471/719
    batch 481/719
    batch 491/719
    batch 501/719
    batch 511/719
    batch 521/719
    batch 531/719
    batch 541/719
    batch 551/719
   

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

    batch 1/719
    batch 11/719
    batch 21/719
    batch 31/719
    batch 41/719
    batch 51/719
    batch 61/719
    batch 71/719
    batch 81/719
    batch 91/719
    batch 101/719
    batch 111/719
    batch 121/719
    batch 131/719
    batch 141/719
    batch 151/719
    batch 161/719
    batch 171/719
    batch 181/719
    batch 191/719
    batch 201/719
    batch 211/719
    batch 221/719
    batch 231/719
    batch 241/719
    batch 251/719
    batch 261/719
    batch 271/719
    batch 281/719
    batch 291/719
    batch 301/719
    batch 311/719
    batch 321/719
    batch 331/719
    batch 341/719
    batch 351/719
    batch 361/719
    batch 371/719
    batch 381/719
    batch 391/719
    batch 401/719
    batch 411/719
    batch 421/719
    batch 431/719
    batch 441/719
    batch 451/719
    batch 461/719
    batch 471/719
    batch 481/719
    batch 491/719
    batch 501/719
    batch 511/719
    batch 521/719
    batch 531/719
    batch 541/719
    batch 551/719
   

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

    batch 1/719
    batch 11/719
    batch 21/719
    batch 31/719
    batch 41/719
    batch 51/719
    batch 61/719
    batch 71/719
    batch 81/719
    batch 91/719
    batch 101/719
    batch 111/719
    batch 121/719
    batch 131/719
    batch 141/719
    batch 151/719
    batch 161/719
    batch 171/719
    batch 181/719
    batch 191/719
    batch 201/719
    batch 211/719
    batch 221/719
    batch 231/719
    batch 241/719
    batch 251/719
    batch 261/719
    batch 271/719
    batch 281/719
    batch 291/719
    batch 301/719
    batch 311/719
    batch 321/719
    batch 331/719
    batch 341/719
    batch 351/719
    batch 361/719
    batch 371/719
    batch 381/719
    batch 391/719
    batch 401/719
    batch 411/719
    batch 421/719
    batch 431/719
    batch 441/719
    batch 451/719
    batch 461/719
    batch 471/719
    batch 481/719
    batch 491/719
    batch 501/719
    batch 511/719
    batch 521/719
    batch 531/719
    batch 541/719
    batch 551/719
   

---

## 3. Approach 1: True Stacking (Meta-Model)

Train a meta-learner on top of OOF predictions from CatBoost and DeBERTa.


In [13]:
# Stack OOF predictions: CatBoost probs (3) + DeBERTa probs (3) = 6 features
stacked_features = np.hstack(
    [
        catboost_oof,  # (n, 3)
        deberta_oof_probs,  # (n, 3)
    ]
)
print(f"Stacked features shape: {stacked_features.shape}")

# --- 3a. Logistic Regression meta-model ---
lr_meta_oof = np.zeros((len(y), 3))
lr_meta_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(CV_SPLITS):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(stacked_features[train_idx])
    X_val = scaler.transform(stacked_features[val_idx])

    meta = LogisticRegression(max_iter=2000, random_state=SEED)
    meta.fit(X_train, y[train_idx])
    pred = meta.predict_proba(X_val)
    lr_meta_oof[val_idx] = pred
    score = log_loss(y[val_idx], pred)
    lr_meta_fold_scores.append(score)
    print(f"  Fold {fold}: {score:.4f}")

lr_meta_loss = log_loss(y, lr_meta_oof)
print(
    f"\nLR Stacking OOF log loss: {lr_meta_loss:.4f} (+/- {np.std(lr_meta_fold_scores):.4f})"
)

# --- 3b. CatBoost meta-model ---
cb_meta_oof = np.zeros((len(y), 3))
cb_meta_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(CV_SPLITS):
    meta = CatBoostClassifier(
        loss_function="MultiClass",
        classes_count=3,
        iterations=300,
        learning_rate=0.05,
        depth=4,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
    )
    meta.fit(stacked_features[train_idx], y[train_idx])
    pred = meta.predict_proba(stacked_features[val_idx])
    cb_meta_oof[val_idx] = pred
    score = log_loss(y[val_idx], pred)
    cb_meta_fold_scores.append(score)
    print(f"  Fold {fold}: {score:.4f}")

cb_meta_loss = log_loss(y, cb_meta_oof)
print(
    f"\nCatBoost Stacking OOF log loss: {cb_meta_loss:.4f} (+/- {np.std(cb_meta_fold_scores):.4f})"
)

Stacked features shape: (57477, 6)
  Fold 0: 0.9922
  Fold 1: 0.9949
  Fold 2: 0.9869
  Fold 3: 0.9887
  Fold 4: 0.9967

LR Stacking OOF log loss: 0.9918 (+/- 0.0037)
  Fold 0: 0.9911
  Fold 1: 0.9938
  Fold 2: 0.9862
  Fold 3: 0.9865
  Fold 4: 0.9953

CatBoost Stacking OOF log loss: 0.9906 (+/- 0.0037)


---

## 4. Approach 2: CLS + Handcrafted Features MLP

Concatenate the `[CLS]` embedding from DeBERTa with the 20 handcrafted features
and train a small MLP end-to-end with stratified CV.


### 4a. MLP HP Tuning (Optuna)

Tune MLP architecture and training hyperparameters using 5-fold stratified CV.
Objective: minimize mean OOF log loss.


In [ ]:
class CLSFeatureMLP(nn.Module):
    def __init__(
        self, cls_dim, feat_dim, num_classes=3, hidden_dims=(512, 256), dropout=0.3
    ):
        super().__init__()
        layers = []
        in_dim = cls_dim + feat_dim
        for h in hidden_dims:
            layers += [
                nn.BatchNorm1d(in_dim),
                nn.Linear(in_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers += [nn.BatchNorm1d(in_dim), nn.Linear(in_dim, num_classes)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp_fold(
    cls_train,
    feat_train,
    y_train,
    cls_val,
    feat_val,
    y_val,
    device,
    hidden_dims=(512, 256),
    dropout=0.3,
    epochs=30,
    batch_size=256,
    lr=1e-3,
    weight_decay=1e-4,
    patience=7,
):
    cls_dim = cls_train.shape[1]
    feat_dim = feat_train.shape[1]

    # Standardize features
    scaler_cls = StandardScaler()
    cls_train_s = scaler_cls.fit_transform(cls_train)
    cls_val_s = scaler_cls.transform(cls_val)

    scaler_feat = StandardScaler()
    feat_train_s = scaler_feat.fit_transform(feat_train)
    feat_val_s = scaler_feat.transform(feat_val)

    X_train_np = np.hstack([cls_train_s, feat_train_s]).astype(np.float32)
    X_val_np = np.hstack([cls_val_s, feat_val_s]).astype(np.float32)

    X_train_t = torch.tensor(X_train_np, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)
    X_val_t = torch.tensor(X_val_np, dtype=torch.float32, device=device)

    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = CLSFeatureMLP(
        cls_dim, feat_dim, num_classes=3, hidden_dims=hidden_dims, dropout=dropout
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t).cpu().numpy()
            val_probs = np.exp(val_logits - val_logits.max(axis=1, keepdims=True))
            val_probs = val_probs / val_probs.sum(axis=1, keepdims=True)
            val_loss = log_loss(y_val, val_probs)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Load best and get final OOF prediction
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t).cpu().numpy()
        val_probs = np.exp(val_logits - val_logits.max(axis=1, keepdims=True))
        val_probs = val_probs / val_probs.sum(axis=1, keepdims=True)

    return val_probs, best_val_loss


print("MLP model and training function defined.")

MLP model and training function defined.


In [ ]:
def mlp_objective(trial):
    hidden_dim_1 = trial.suggest_int("hidden_dim_1", 64, 512)
    hidden_dim_2 = trial.suggest_int("hidden_dim_2", 32, hidden_dim_1)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
    patience = trial.suggest_int("patience", 5, 10)
    epochs = 50

    hidden_dims = (hidden_dim_1, hidden_dim_2)
    oof = np.zeros((len(y), 3))

    for train_idx, val_idx in CV_SPLITS:
        val_probs, _ = train_mlp_fold(
            cls_train=deberta_oof_cls[train_idx],
            feat_train=X_20[train_idx],
            y_train=y[train_idx],
            cls_val=deberta_oof_cls[val_idx],
            feat_val=X_20[val_idx],
            y_val=y[val_idx],
            device=DEVICE,
            hidden_dims=hidden_dims,
            dropout=dropout,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            patience=patience,
        )
        oof[val_idx] = val_probs

    return log_loss(y, oof)


mlp_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
mlp_study.optimize(mlp_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nBest trial:")
print(f"  Log loss: {mlp_study.best_trial.value:.6f}")
print(f"  Params:   {mlp_study.best_trial.params}")

In [ ]:
# Run MLP OOF with CLS + handcrafted features using best HPs from Optuna
best_mlp_params = mlp_study.best_trial.params
best_hidden_dims = (best_mlp_params["hidden_dim_1"], best_mlp_params["hidden_dim_2"])
print("Training MLP with CLS + handcrafted features...")
print(
    f"Best HPs: hidden_dims={best_hidden_dims}, dropout={best_mlp_params['dropout']:.4f}, "
    f"lr={best_mlp_params['lr']:.6f}, weight_decay={best_mlp_params['weight_decay']:.6f}, "
    f"batch_size={best_mlp_params['batch_size']}, patience={best_mlp_params['patience']}"
)

mlp_oof = np.zeros((len(y), 3))
mlp_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(CV_SPLITS):
    print(f"\n--- Fold {fold} ---")
    val_probs, val_loss = train_mlp_fold(
        cls_train=deberta_oof_cls[train_idx],
        feat_train=X_20[train_idx],
        y_train=y[train_idx],
        cls_val=deberta_oof_cls[val_idx],
        feat_val=X_20[val_idx],
        y_val=y[val_idx],
        device=DEVICE,
        hidden_dims=best_hidden_dims,
        dropout=best_mlp_params["dropout"],
        epochs=50,
        batch_size=best_mlp_params["batch_size"],
        lr=best_mlp_params["lr"],
        weight_decay=best_mlp_params["weight_decay"],
        patience=best_mlp_params["patience"],
    )
    mlp_oof[val_idx] = val_probs
    mlp_fold_scores.append(val_loss)
    print(f"  Fold {fold} log loss: {val_loss:.4f}")

mlp_oof_loss = log_loss(y, mlp_oof)
print(
    f"\nMLP (CLS + features) OOF log loss: {mlp_oof_loss:.4f} (+/- {np.std(mlp_fold_scores):.4f})"
)

Training MLP with CLS + handcrafted features...
Best HPs: hidden_dims=(257, 97), dropout=0.3447, lr=0.000190, weight_decay=0.000038, batch_size=512, patience=6

--- Fold 0 ---
  Fold 0 log loss: 0.9885

--- Fold 1 ---
  Fold 1 log loss: 0.9915

--- Fold 2 ---
  Fold 2 log loss: 0.9841

--- Fold 3 ---
  Fold 3 log loss: 0.9842

--- Fold 4 ---
  Fold 4 log loss: 0.9918

MLP (CLS + features) OOF log loss: 0.9880 (+/- 0.0034)


/home/danil/Documents/PMDL/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:259: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


---

## 5. Final Report


In [21]:
results = pd.DataFrame(
    [
        {
            "approach": "CatBoost (20 features, HP-tuned)",
            "log_loss": catboost_oof_loss,
            "fold_std": np.std(catboost_fold_scores),
        },
        {
            "approach": "DeBERTa-v3 cross-encoder",
            "log_loss": deberta_oof_loss,
            "fold_std": 0.0,  # computed per-fold above
        },
        {
            "approach": "Stacking: LR meta-model",
            "log_loss": lr_meta_loss,
            "fold_std": np.std(lr_meta_fold_scores),
        },
        {
            "approach": "Stacking: CatBoost meta-model",
            "log_loss": cb_meta_loss,
            "fold_std": np.std(cb_meta_fold_scores),
        },
        {
            "approach": "MLP (CLS + 20 handcrafted features)",
            "log_loss": mlp_oof_loss,
            "fold_std": np.std(mlp_fold_scores),
        },
    ]
)

results = results.sort_values("log_loss")
results["log_loss"] = results["log_loss"].map("{:.4f}".format)
results["fold_std"] = results["fold_std"].map("{:.4f}".format)

print("=" * 60)
print("FINAL RESULTS (OOF Log Loss, 5-fold stratified CV)")
print("=" * 60)
display(results)

print(f"\nBest CatBoost HPs: {study.best_trial.params}")
print(f"Best MLP HPs:     {mlp_study.best_trial.params}")

FINAL RESULTS (OOF Log Loss, 5-fold stratified CV)


,approach,log_loss,fold_std
4,MLP (CLS + 20 handcrafted features),0.9880,0.0034
3,Stacking: CatBoost meta-model,0.9906,0.0037
2,Stacking: LR meta-model,0.9918,0.0037
1,DeBERTa-v3 cross-encoder,0.9939,0.0000
0,"CatBoost (20 features, HP-tuned)",1.0279,0.0026



Best CatBoost HPs: {'iterations': 896, 'learning_rate': 0.028869220380495747, 'depth': 7, 'l2_leaf_reg': 5.920392514089517, 'border_count': 73}
Best MLP HPs:     {'hidden_dim_1': 257, 'hidden_dim_2': 97, 'dropout': 0.34474115788895177, 'lr': 0.00019010245319870352, 'weight_decay': 3.8396292998041685e-05, 'batch_size': 512, 'patience': 6}
